In [41]:
import pandas as pd
import glob
import os

# Get 5 CSV files from the folder
cgm_5files_hg = glob.glob("C:/Users/Owner/Downloads/Data_5file/*.csv")

# Add Patient_ID column & merge all 5 files
dfs = []
for file in cgm_5files_hg:
    if os.path.isfile(file):
        df = pd.read_csv(file, sep=';')
        df['patient_id'] = os.path.basename(file).replace('.csv', '')
        dfs.append(df)   # ← this must be inside the IF block

# Merge all files
df_CGM5_hg = pd.concat(dfs, ignore_index=True)

# Move patient_id to first column
cols = ['patient_id'] + [col for col in df_CGM5_hg.columns if col != 'patient_id']
df_CGM5_hg = df_CGM5_hg[cols]

df_CGM5_hg

,patient_id,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,HUPA0019P,2019-07-03T14:00:00,195.000000,6.14906,60.172414,0.0,0.069,1.0,1.0
1,HUPA0019P,2019-07-03T14:05:00,193.666667,9.74568,64.264706,20.0,0.069,0.0,0.0
2,HUPA0019P,2019-07-03T14:10:00,192.333333,12.76220,75.464286,50.0,0.069,0.0,0.0
3,HUPA0019P,2019-07-03T14:15:00,191.000000,14.96658,76.360000,65.0,0.069,0.0,0.0
4,HUPA0019P,2019-07-03T14:20:00,192.666667,9.28160,76.172414,13.0,0.069,0.0,0.0
...,...,...,...,...,...,...,...,...,...
16853,HUPA0023P,2020-01-30T14:10:00,76.000000,6.72650,67.931818,0.0,0.035,0.0,0.0
16854,HUPA0023P,2020-01-30T14:15:00,76.000000,12.84150,69.116279,39.0,0.035,0.0,0.0
16855,HUPA0023P,2020-01-30T14:20:00,76.333333,6.60420,63.785714,0.0,0.035,0.0,0.0
16856,HUPA0023P,2020-01-30T14:25:00,76.666667,6.84880,69.115385,0.0,0.035,0.0,0.0


In [42]:
# Convert all column names to title case
df_CGM5_hg.columns = df_CGM5_hg.columns.str.title()

# Check Existing datatype of columns
df_CGM5_hg.dtypes

# Convert 'time' column from object to datetime format for time-based operation
df_CGM5_hg['Time'] = pd.to_datetime(df_CGM5_hg['Time'])

# Convert 'steps' column from float64 to int as steps are always whole numbers
df_CGM5_hg['Steps'] = df_CGM5_hg['Steps'].astype(int)

df_CGM5_hg.dtypes

Patient_Id                        object
Time                      datetime64[ns]
Glucose                          float64
Calories                         float64
Heart_Rate                       float64
Steps                              int64
Basal_Rate                       float64
Bolus_Volume_Delivered           float64
Carb_Input                       float64
dtype: object

In [48]:
# Round all float64 columns to 2 decimal places
float_cols = df_CGM5_hg.select_dtypes(include='float64').columns
df_CGM5_hg[float_cols] = df_CGM5_hg[float_cols].round(2)


df_CGM5_hg.tail()

,Patient_Id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
16853,HUPA0023P,2020-01-30 14:10:00,76.00,6.73,67.93,0,0.04,0.0,0.0
16854,HUPA0023P,2020-01-30 14:15:00,76.00,12.84,69.12,39,0.04,0.0,0.0
16855,HUPA0023P,2020-01-30 14:20:00,76.33,6.60,63.79,0,0.04,0.0,0.0
16856,HUPA0023P,2020-01-30 14:25:00,76.67,6.85,69.12,0,0.04,0.0,0.0
16857,HUPA0023P,2020-01-30 14:30:00,77.00,12.96,71.15,20,0.04,0.0,0.0


In [43]:
#Duplicate value check
print("Total duplicates:", df_CGM5_hg.duplicated().sum())

Total duplicates: 0


In [44]:
#Null Value
df_CGM5_hg.isnull().sum()

Patient_Id                0
Time                      0
Glucose                   0
Calories                  0
Heart_Rate                0
Steps                     0
Basal_Rate                0
Bolus_Volume_Delivered    0
Carb_Input                0
dtype: int64

In [49]:
# No column should have negative values
numeric_cols = df_CGM5_hg.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    neg = (df_CGM5_hg[col] < 0).sum()
    print(f"{col}: {neg} negative values")

Glucose: 0 negative values
Calories: 0 negative values
Heart_Rate: 0 negative values
Steps: 0 negative values
Basal_Rate: 0 negative values
Bolus_Volume_Delivered: 0 negative values
Carb_Input: 0 negative values


In [50]:
#check white space
mask = df_CGM5_hg.apply(lambda col: col.astype(str).str.contains(r"^\s+|\s+$"))
df_CGM5_hg[mask.any(axis=1)]

,Patient_Id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input


In [51]:
# Sort data by patient and time for proper time series analysis
df_CGM5_hg = df_CGM5_hg.sort_values(['Patient_Id', 'Time']).reset_index(drop=True)
df_CGM5_hg.head()

,Patient_Id,Time,Glucose,Calories,Heart_Rate,Steps,Basal_Rate,Bolus_Volume_Delivered,Carb_Input
0,HUPA0019P,2019-07-03 14:00:00,195.00,6.15,60.17,0,0.07,1.0,1.0
1,HUPA0019P,2019-07-03 14:05:00,193.67,9.75,64.26,20,0.07,0.0,0.0
2,HUPA0019P,2019-07-03 14:10:00,192.33,12.76,75.46,50,0.07,0.0,0.0
3,HUPA0019P,2019-07-03 14:15:00,191.00,14.97,76.36,65,0.07,0.0,0.0
4,HUPA0019P,2019-07-03 14:20:00,192.67,9.28,76.17,13,0.07,0.0,0.0


In [53]:
df_CGM5_hg.to_csv(
    r"C:/Users/Owner/Desktop/Harshali/DataAnalyst-Numpy Ninja/Python_Hakython/Team2_-PyQueens_-Python-Hackathon-_MAY-2026/Harshali_Datacleaning/Clean-data-fivefile.csv",
    index=False
)
print("Cleaned .csv file created")

Cleaned .csv file created
